# Detecção e Tratamento de Outliers
## Identificação de Valores Atípicos

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 06

---

# Parte I: Conceitos Teóricos

# =====================================
# 1. INTRODUÇÃO AOS OUTLIERS
# =====================================

OBJETIVOS DA AULA:
- Compreender o conceito de outliers e seu impacto na análise de dados
- Identificar diferentes tipos de outliers
- Aprender métodos para detecção de outliers
- Implementar técnicas para tratamento de outliers
- Avaliar o impacto das diferentes estratégias de tratamento

In [ ]:
# @title
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ==================================================
# 2. DEFINIÇÃO E IMPORTÂNCIA DOS OUTLIERS
# ==================================================

O QUE SÃO OUTLIERS?

Outliers são observações que apresentam um **grande afastamento** das demais da série,
ou que são **inconsistentes**. Em outras palavras, outlier é um valor atípico, um valor
que "foge" da normalidade e que pode (e provavelmente irá) causar anomalias nos
resultados obtidos por meio de algoritmos e sistemas de análise.

IMPORTÂNCIA DA DETECÇÃO E TRATAMENTO:

1. Impacto na Análise:
   - Distorcem medidas estatísticas (média, desvio padrão)
   - Afetam a distribuição dos dados
   - Influenciam coeficientes de correlação e regressão
   - Podem levar a conclusões incorretas

2. Impacto em Algoritmos de Machine Learning:
   - Modelos lineares são altamente sensíveis a outliers
   - Algoritmos baseados em distância são impactados
   - Podem causar overfitting ou underfitting
   - Reduzem a capacidade de generalização

3. Valor dos Outliers:
   - Nem sempre devem ser removidos
   - Podem conter informações importantes
   - Podem indicar erros de medição ou registro
   - Podem revelar comportamentos ou eventos raros relevantes

overfitting-underfitting.svg

# ==================================================
# 3. CRIANDO UM CONJUNTO DE DADOS COM OUTLIERS
# ==================================================

In [ ]:
# @title
# Gerando um dataset sintético com diferentes tipos de outliers
np.random.seed(42)
n = 1000

# Base de dados: imóveis residenciais para venda
# Criando dados para as principais características
data = {
    'area_m2': np.random.normal(120, 30, n),
    'quartos': np.random.choice([1, 2, 3, 4, 5], n, p=[0.1, 0.3, 0.4, 0.15, 0.05]),
    'banheiros': np.random.choice([1, 2, 3, 4], n, p=[0.2, 0.5, 0.2, 0.1]),
    'idade_anos': np.random.gamma(4, 5, n),
    'dist_centro_km': np.random.gamma(2, 3, n),
    'area_terreno_m2': np.random.normal(300, 100, n),
    'garagem_vagas': np.random.choice([0, 1, 2, 3, 4], n, p=[0.1, 0.2, 0.5, 0.15, 0.05]),
    'qualidade_acabamento': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.15, 0.5, 0.2, 0.1]),
}

In [ ]:
# @title
# Criação do DataFrame
df_imoveis = pd.DataFrame(data)

# Calculando o preço com base nas características (e adicionando ruído)
df_imoveis['preco'] = (
    200000 +                                    # preço base
    df_imoveis['area_m2'] * 3000 +              # valor por m²
    df_imoveis['quartos'] * 50000 +             # valor por quarto
    df_imoveis['banheiros'] * 40000 +           # valor por banheiro
    df_imoveis['garagem_vagas'] * 30000 -       # valor por vaga
    df_imoveis['idade_anos'] * 2000 -           # desconto por idade
    df_imoveis['dist_centro_km'] * 10000 +      # desconto por distância
    df_imoveis['qualidade_acabamento'] * 60000  # adicional por qualidade
) + np.random.normal(0, 100000, n)              # ruído aleatório

df_imoveis.head()


# ==================================================
# 4. CLASSIFICAÇÃO DE OUTLIERS
# ==================================================

## TIPOS DE OUTLIERS:

Outliers podem ser classificados de acordo com sua natureza e dimensionalidade:

### 1. **Outliers Univariados**
   - Valores extremos em uma única variável
   - Exemplo: Uma pessoa com 2.5m de altura em um dataset de alturas
   - Detectados com: Boxplot, Z-score, IQR, Percentis
   
### 2. **Outliers Bivariados**
   - Valores anormais na relação entre duas variáveis
   - Exemplo: Uma casa pequena com preço muito alto
   - Podem ser normais individualmente, mas anormais em conjunto
   - Detectados com: Scatter plots, elipses de confiança
   
### 3. **Outliers Multivariados**
   - Anormalidades considerando múltiplas dimensões simultaneamente
   - Difíceis de visualizar diretamente
   - Detectados com: Distância de Mahalanobis, PCA, métodos de clustering

### **Por que classificar?**
- Diferentes tipos requerem diferentes métodos de detecção
- A estratégia de tratamento pode variar conforme o tipo
- Ajuda a entender a natureza do problema nos dados

In [ ]:
# @title Boxplots para Outliers Univariados
# 1. Boxplots para Outliers Univariados
numerical_cols = ['area_m2', 'idade_anos', 'dist_centro_km', 'area_terreno_m2', 'preco']

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 2, i)
    sns.boxplot(y=df_imoveis[col])
    plt.title(f'Boxplot de {col}')
    plt.tight_layout()
plt.show()

In [ ]:
# @title Histogramas para visualizar distribuições
# 2. Histogramas para visualizar distribuições
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 2, i)
    sns.histplot(df_imoveis[col], kde=True)
    plt.title(f'Histograma de {col}')
    plt.tight_layout()
plt.show()

In [ ]:
# @title Scatter plots para outliers multivariados
# 3. Scatter plots para outliers multivariados
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(x='area_m2', y='preco', data=df_imoveis, alpha=0.6)
plt.title('Preço vs Área')

plt.subplot(1, 3, 2)
sns.scatterplot(x='quartos', y='area_m2', data=df_imoveis, alpha=0.6)
plt.title('Área vs Quartos')

plt.subplot(1, 3, 3)
sns.scatterplot(x='idade_anos', y='preco', data=df_imoveis, alpha=0.6)
plt.title('Preço vs Idade')

plt.tight_layout()
plt.show()

In [ ]:
# @title Pairplot para visualizar relações entre múltiplas variáveis
g = sns.pairplot(df_imoveis[['area_m2', 'quartos', 'preco', 'idade_anos']], height=3)

# Adicionando títulos na parte superior de cada gráfico
variables = ['area_m2', 'quartos', 'preco', 'idade_anos']
titles = ['Área (m²)', 'Quartos', 'Preço', 'Idade (anos)']  # Rótulos mais informativos

# Adicionar títulos na parte superior de cada coluna
for j, title in enumerate(titles):
    g.axes[0, j].set_title(title)

plt.suptitle('Matriz de Dispersão para Variáveis Selecionadas', y=1.02)
plt.tight_layout()
plt.show()

# ==================================================
# 5. MÉTODOS ESTATÍSTICOS PARA DETECÇÃO
# ==================================================

## VISÃO GERAL DOS MÉTODOS ESTATÍSTICOS:

Os métodos estatísticos para detecção de outliers baseiam-se em propriedades matemáticas da distribuição dos dados:

### **Métodos Paramétricos:**
- Assumem uma distribuição específica (geralmente normal)
- Exemplos: Z-score, Grubbs test
- Vantagem: Simples e rápidos
- Desvantagem: Sensíveis a violações das suposições

### **Métodos Não-Paramétricos:**
- Não assumem distribuição específica
- Exemplos: IQR, MAD (Median Absolute Deviation)
- Vantagem: Robustos e flexíveis
- Desvantagem: Podem ser menos sensíveis

### **Critérios de Decisão:**
- Limiar estatístico (ex: |Z| > 3)
- Percentis (ex: < 1% ou > 99%)
- Múltiplos do IQR (ex: 1.5 × IQR)

A seguir, exploraremos os principais métodos em detalhe.

# ==================================================
# 6. MÉTODO Z-SCORE
# ==================================================

**Z-Score** mede quantos desvios padrão uma observação está acima ou abaixo da média.

Fórmula: Z = (X - μ) / σ

Onde:
- X é o valor observado
- μ é a média
- σ é o desvio padrão

**Interpretação:**
- |Z| > 3: Potencial outlier (99.7% dos dados em uma distribuição normal estão dentro de 3 desvios padrão)
- |Z| > 2: Possível outlier (95% dos dados em uma distribuição normal estão dentro de 2 desvios padrão)

**Limitações:**
- Assume distribuição aproximadamente normal
- A média e o desvio padrão são influenciados pelos próprios outliers
- Não funciona bem com datasets pequenos

In [ ]:
# @title Calculando Z-Score para as variáveis numéricas
z_scores = {}
for col in numerical_cols:
    z_scores[col] = stats.zscore(df_imoveis[col])

# Criando DataFrame com Z-Scores
df_z = pd.DataFrame(z_scores)

# Visualizando os Z-Scores mais extremos
print("Z-Scores mais extremos:")
#print(df_z.apply(lambda x: x.abs().sort_values(ascending=False).head()))
print(df_z.apply(lambda x: x.abs().sort_values(ascending=False).head()).fillna(''))

In [ ]:
# @title Identificando outliers baseado no Z-Score
threshold = 3
outliers_z_score = {}

for col in numerical_cols:
    # Índices onde |Z| > threshold
    outlier_indices = np.where(np.abs(z_scores[col]) > threshold)[0]
    outliers_z_score[col] = {
        'count': len(outlier_indices),
        'percentage': len(outlier_indices) / len(df_imoveis) * 100,
        'indices': outlier_indices,
        'values': df_imoveis.loc[outlier_indices, col].values
    }

    print(f"\nOutliers em {col} usando Z-Score (|Z| > {threshold}):")
    print(f"  - Quantidade: {outliers_z_score[col]['count']}")
    print(f"  - Percentual: {outliers_z_score[col]['percentage']:.2f}%")
    if len(outlier_indices) > 0:
        print(f"  - Valores: {outliers_z_score[col]['values'][:5]}{'...' if len(outlier_indices) > 5 else ''}")

In [ ]:
# @title Visualizando a distribuição dos Z-Scores
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 2, i)
    sns.histplot(z_scores[col], kde=True)

    # Linhas verticais para os thresholds
    plt.axvline(x=threshold, color='red', linestyle='--', alpha=0.7, label=f'Z = {threshold}')
    plt.axvline(x=-threshold, color='red', linestyle='--', alpha=0.7)
    plt.axvline(x=2, color='orange', linestyle=':', alpha=0.7, label='Z = 2')
    plt.axvline(x=-2, color='orange', linestyle=':', alpha=0.7)

    plt.title(f'Distribuição Z-Score de {col}')
    plt.legend()

plt.tight_layout()
plt.show()

# ==================================================
# 7. MÉTODO MAD (MEDIAN ABSOLUTE DEVIATION)
# ==================================================

O **Z-Score Modificado** usa a mediana e o Desvio Absoluto da Mediana (MAD) em vez da média e desvio padrão.

MAD: Mediana dos desvios absolutos da mediana.

Fórmula: $M_i = 0.6745 \times \frac{(x_i - mediana)}{MAD}$

Onde:
- $x_i$ é o valor observado
- $mediana$ é a mediana dos dados
- $MAD$ é a mediana dos desvios absolutos da mediana: $MAD = mediana(|x_i - mediana|)$
- O fator 0.6745 torna o MAD comparável com o desvio padrão para dados normais

**Interpretação:**
- |M| > 3.5: Potencial outlier (Critério de Iglewicz & Hoaglin)

**Vantagens:**
- Mais robusto que o Z-Score tradicional
- Menos influenciado pelos próprios outliers

In [ ]:
# @title
# Função para calcular Z-Score Modificado
def modified_z_score(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))  # MAD
    if mad == 0:  # Evitar divisão por zero
        return np.zeros_like(data)
    return 0.6745 * (data - median) / mad

In [ ]:
# @title
# Calculando Z-Score Modificado para as variáveis numéricas
mod_z_scores = {}
for col in numerical_cols:
    mod_z_scores[col] = modified_z_score(df_imoveis[col].values)

# Criando DataFrame com Z-Scores Modificados
df_mod_z = pd.DataFrame(mod_z_scores)

# Visualizando os Z-Scores Modificados mais extremos
print("Z-Scores Modificados mais extremos:")
print(df_mod_z.apply(lambda x: x.abs().sort_values(ascending=False).head()).fillna(''))

In [ ]:
# @title Identificando outliers baseado no Z-Score Modificado
threshold_mod = 3.5  # Critério de Iglewicz e Hoaglin
outliers_mod_z = {}

for col in numerical_cols:
    # Índices onde |M| > threshold_mod
    outlier_indices = np.where(np.abs(mod_z_scores[col]) > threshold_mod)[0]
    outliers_mod_z[col] = {
        'count': len(outlier_indices),
        'percentage': len(outlier_indices) / len(df_imoveis) * 100,
        'indices': outlier_indices,
        'values': df_imoveis.loc[outlier_indices, col].values
    }

    print(f"\nOutliers em {col} usando Z-Score Modificado (|M| > {threshold_mod}):")
    print(f"  - Quantidade: {outliers_mod_z[col]['count']}")
    print(f"  - Percentual: {outliers_mod_z[col]['percentage']:.2f}%")
    if len(outlier_indices) > 0:
        print(f"  - Valores: {outliers_mod_z[col]['values'][:5]}{'...' if len(outlier_indices) > 5 else ''}")

In [ ]:
# @title
# Comparando resultados: Z-Score vs Z-Score Modificado
comparison = {}
for col in numerical_cols:
    comparison[col] = {
        'Z-Score Count': outliers_z_score[col]['count'],
        'Modified Z-Score Count': outliers_mod_z[col]['count'],
        'Difference': outliers_mod_z[col]['count'] - outliers_z_score[col]['count']
    }

comparison_df = pd.DataFrame(comparison).T
print("\nComparação entre métodos:")
print(comparison_df)

# ==================================================
# 8. MÉTODO IQR (INTERVALO INTERQUARTIL)
# ==================================================

O **método IQR** (Intervalo Interquartil) usa quartis para identificar outliers.

Definições:
- Q1 = 1º quartil (25% dos dados)
- Q3 = 3º quartil (75% dos dados)
- IQR = Q3 - Q1 (Intervalo Interquartil)

Limite inferior: Q1 - k * IQR
Limite superior: Q3 + k * IQR

Onde k é uma constante (geralmente 1.5 ou 3):
- k = 1.5: Outliers possíveis/suspeitos
- k = 3: Outliers extremos

**Vantagens:**
- Não assume normalidade dos dados
- Robusto a valores extremos
- Método usado em boxplots

In [ ]:
# @title
# Função para detecção de outliers usando IQR
def detect_outliers_iqr(data, k=1.5):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - k * IQR
    upper_bound = Q3 + k * IQR

    outliers = (data < lower_bound) | (data > upper_bound)

    return {
        'outliers': outliers,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'count': outliers.sum(),
        'percentage': outliers.sum() / len(data) * 100,
        'indices': outliers[outliers].index.tolist(),
        'values': data[outliers].values
    }

# Aplicando IQR para todas as variáveis com diferentes valores de k
outliers_iqr = {}

for col in numerical_cols:
    for k in [1.5, 3.0]:
        key = f"{col}_k{k}"
        result = detect_outliers_iqr(df_imoveis[col], k=k)
        outliers_iqr[key] = result

        print(f"\nOutliers em {col} usando IQR (k={k}):")
        print(f"  - Quantidade: {result['count']}")
        print(f"  - Percentual: {result['percentage']:.2f}%")
        print(f"  - Limite inferior: {result['lower_bound']:.2f}")
        print(f"  - Limite superior: {result['upper_bound']:.2f}")

In [ ]:
# @title Visualizando a aplicação do método IQR
k_values = [1.5, 3.0]  # Comparando k=1.5 e k=3.0

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 2, i)
    sns.histplot(df_imoveis[col], kde=True)

    for k in k_values:
        key = f"{col}_k{k}"
        # Linhas verticais para os limites IQR
        plt.axvline(x=outliers_iqr[key]['lower_bound'],
                    color='red' if k==1.5 else 'blue',
                    linestyle='--',
                    label=f'Limites IQR (k={k})')
        plt.axvline(x=outliers_iqr[key]['upper_bound'],
                    color='red' if k==1.5 else 'blue',
                    linestyle='--')

    plt.title(f'Distribuição de {col} com limites IQR')
    plt.legend()

plt.tight_layout()
plt.show()


# ==================================================
# 9. COMPARAÇÃO DE MÉTODOS ESTATÍSTICOS
# ==================================================

In [ ]:
# @title
# Comparando os três métodos para cada variável
method_comparison = {}

for col in numerical_cols:
    method_comparison[col] = {
        'Z-Score (|Z| > 3)': outliers_z_score[col]['count'],
        'Modified Z-Score (|M| > 3.5)': outliers_mod_z[col]['count'],
        'IQR (k = 1.5)': outliers_iqr[f"{col}_k1.5"]['count'],
        'IQR (k = 3.0)': outliers_iqr[f"{col}_k3.0"]['count']
    }

# Criando DataFrame com a comparação
method_comparison_df = pd.DataFrame(method_comparison).T
print("Comparação de métodos de detecção de outliers:")
print(method_comparison_df)

In [ ]:
# @title
# Criando a variável outlier_count como float para evitar problemas de tipo
df_imoveis['outlier_count'] = 0.0

# Contando detecções por Z-Score
for col in numerical_cols:
    outlier_mask = np.abs(stats.zscore(df_imoveis[col])) > 3
    df_imoveis.loc[outlier_mask, 'outlier_count'] += 1

# Resetando para contar todos os métodos adequadamente
df_imoveis['outlier_count'] = 0.0

# Método 1: Z-Score (|Z| > 3)
for col in numerical_cols:
    z = np.abs(stats.zscore(df_imoveis[col]))
    df_imoveis.loc[z > 3, 'outlier_count'] += 1/len(numerical_cols)

# Método 2: Modified Z-Score (|M| > 3.5)
for col in numerical_cols:
    mod_z = np.abs(modified_z_score(df_imoveis[col].values))
    df_imoveis.loc[mod_z > 3.5, 'outlier_count'] += 1/len(numerical_cols)

# Método 3: IQR k=1.5
for col in numerical_cols:
    result = outliers_iqr[f"{col}_k1.5"]
    df_imoveis.loc[result['outliers'], 'outlier_count'] += 1/len(numerical_cols)

# Método 4: IQR k=3.0
for col in numerical_cols:
    result = outliers_iqr[f"{col}_k3.0"]
    df_imoveis.loc[result['outliers'], 'outlier_count'] += 1/len(numerical_cols)

# Arredondando para obter contagem de métodos
df_imoveis['outlier_count'] = (df_imoveis['outlier_count'] * len(numerical_cols)).round()

print(f"Distribuição de outliers detectados por quantidade de métodos:")
print(df_imoveis['outlier_count'].value_counts().sort_index())

In [ ]:
# @title
# Visualizando a comparação
plt.figure(figsize=(14, 8))
method_comparison_df.plot(kind='bar')
plt.title('Número de Outliers Detectados por Método e Variável')
plt.ylabel('Número de Outliers')
plt.xlabel('Variável')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# @title
# Identificando os outliers detectados por todos os métodos
strong_outliers = df_imoveis[df_imoveis['outlier_count'] == 4].copy()
print(f"\nOutliers detectados por todos os 4 métodos ({len(strong_outliers)} observações):")
strong_outliers[numerical_cols].head(10)

In [ ]:
# @title
# Preparando dados para PCA
X_numerical = df_imoveis[numerical_cols]
X_scaled = StandardScaler().fit_transform(X_numerical)

In [ ]:
# @title Visualizando características dos outliers mais fortes em comparação com dados normais
# Boxplot lado a lado para cada variável
plt.figure(figsize=(15, 12))

for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 2, i)

    # Criando um DataFrame auxiliar para o plot
    plot_df = pd.DataFrame({
        'valor': df_imoveis[col],
        'grupo': ['Outlier' if x == 4 else 'Normal' for x in df_imoveis['outlier_count']]
    })

    sns.boxplot(x='grupo', y='valor', data=plot_df)
    plt.title(f'Comparação de {col}')
    plt.xlabel('')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# ==================================================
# 10. VISUALIZAÇÃO DE OUTLIERS
# ==================================================

In [ ]:
# @title
# Utilizando PCA para visualizar os outliers em 2D
from sklearn.decomposition import PCA

# Aplicando PCA para reduzir para 2 dimensões
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Criando um DataFrame para visualização
pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'outlier_count': df_imoveis['outlier_count']
})

# Explicação da variância
explained_variance = pca.explained_variance_ratio_
print(f"Variância explicada pelas componentes principais:")
print(f"  - PC1: {explained_variance[0]*100:.2f}%")
print(f"  - PC2: {explained_variance[1]*100:.2f}%")
print(f"  - Total: {sum(explained_variance)*100:.2f}%")

In [ ]:
# @title
# Visualizando os outliers em 2D
plt.figure(figsize=(12, 8))

# Cores para diferentes níveis de detecção
colors = ['lightblue', 'orange', 'green', 'purple', 'red']
labels = ['0 métodos', '1 método', '2 métodos', '3 métodos', '4 métodos']

# Plotando cada grupo
for i in range(5):
    mask = pca_df['outlier_count'] == i
    plt.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                c=colors[i], label=labels[i], alpha=0.7, edgecolors='w', linewidth=0.5)

plt.title('Visualização de Outliers usando PCA')
plt.xlabel(f'PC1 ({explained_variance[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({explained_variance[1]*100:.2f}%)')
plt.legend(title='Detecção de Outliers')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# @title Exemplificando a remoção de outliers utilizando diferentes critérios

# Método 1: Remover com base no consenso de métodos
# Removendo observações detectadas por 3 ou mais métodos
df_removed_consensus = df_imoveis[df_imoveis['outlier_count'] < 3].copy()

# Método 2: Remover com base em Z-Score
# Criando uma nova cópia do DataFrame original
df_removed_zscore = df_imoveis.copy()
for col in numerical_cols:
    # Calculando Z-Scores e removendo |Z| > 3
    z = np.abs(stats.zscore(df_removed_zscore[col]))
    df_removed_zscore = df_removed_zscore[z < 3]

# Método 3: Remover com base no IQR
df_removed_iqr = df_imoveis.copy()
for col in numerical_cols:
    Q1 = df_removed_iqr[col].quantile(0.25)
    Q3 = df_removed_iqr[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_removed_iqr = df_removed_iqr[(df_removed_iqr[col] >= lower_bound) &
                                    (df_removed_iqr[col] <= upper_bound)]

# Comparando os tamanhos resultantes
print(f"Tamanho do dataset original: {len(df_imoveis)}")
print(f"Após remoção por consenso (3+ métodos): {len(df_removed_consensus)} ({len(df_removed_consensus)/len(df_imoveis)*100:.2f}%)")
print(f"Após remoção baseada em Z-Score: {len(df_removed_zscore)} ({len(df_removed_zscore)/len(df_imoveis)*100:.2f}%)")
print(f"Após remoção baseada em IQR: {len(df_removed_iqr)} ({len(df_removed_iqr)/len(df_imoveis)*100:.2f}%)")

In [ ]:
# @title Comparando as distribuições antes e depois da remoção
plt.figure(figsize=(15, 10))

for i, col in enumerate(numerical_cols[:4], 1):  # Limitando a 4 variáveis para melhor visualização
    plt.subplot(2, 2, i)

    # Original
    sns.kdeplot(df_imoveis[col], label='Original', color='blue')

    # Após remoção por consenso
    sns.kdeplot(df_removed_consensus[col], label='Após Consenso', color='green')

    # Após remoção com Z-Score
    sns.kdeplot(df_removed_zscore[col], label='Após Z-Score', color='red')

    # Após remoção com IQR
    sns.kdeplot(df_removed_iqr[col], label='Após IQR', color='purple')

    plt.title(f'Distribuição de {col}')
    plt.legend()

plt.tight_layout()
plt.show()

# ==================================================
# 11. TRATAMENTO DE OUTLIERS
# ==================================================

## Transformações
As transformações de variáveis são uma estratégia frequentemente utilizada para lidar com outliers e assimetrias na distribuição dos dados. Em vez de remover as observações, modificamos a escala ou distribuição dos dados.

Principais transformações para lidar com outliers:

1. **Transformação Logarítmica**
   - Aplica log(x) ou log(x+constante) para valores positivos
   - Muito eficaz para dados com distribuição assimétrica positiva
   - Comprime valores grandes e expande valores pequenos
   - Fórmula: X_log = log(X + c), onde c evita log(0)

2. **Transformação Raiz Quadrada**
   - Menos drástica que a transformação logarítmica
   - Útil para dados de contagem e variáveis positivamente assimétricas
   - Fórmula: X_sqrt = sqrt(X)

3. **Transformação Box-Cox**
   - Família de transformações com parâmetro λ (lambda)
   - Encontra automaticamente o melhor parâmetro de transformação
   - Fórmula:
     - λ ≠ 0: (X^λ - 1)/λ
     - λ = 0: log(X)
   - Requer valores estritamente positivos

In [ ]:
# @title
# Aplicando transformações às variáveis para lidar com outliers

# Criando DataFrames para diferentes transformações
df_transformed = df_imoveis.copy()

# 1. Transformação logarítmica (log(x+1) para lidar com zeros)
for col in numerical_cols:
    # Garantir que todos os valores sejam positivos
    min_val = df_transformed[col].min()
    if min_val <= 0:
        offset = abs(min_val) + 1
    else:
        offset = 0

    df_transformed[f"{col}_log"] = np.log1p(df_transformed[col] + offset)

# 2. Transformação raiz quadrada
for col in numerical_cols:
    # Garantir que todos os valores sejam positivos
    min_val = df_transformed[col].min()
    if min_val < 0:
        offset = abs(min_val) + 0.01
    else:
        offset = 0

    df_transformed[f"{col}_sqrt"] = np.sqrt(df_transformed[col] + offset)

## Winsorização (Capping/Clipping)

A **Winsorização** é uma técnica de tratamento de outliers que **limita valores extremos** sem removê-los completamente. Em vez de excluir outliers, substituímos os valores extremos pelos valores de percentis específicos.

**Como funciona:**
- Define-se limites baseados em percentis (ex: 5º e 95º percentil)
- Valores abaixo do limite inferior são substituídos pelo valor do limite inferior
- Valores acima do limite superior são substituídos pelo valor do limite superior

**Exemplo prático:**
Se o 5º percentil = R\$ 100.000 e o 95º percentil = R\$ 900.000:

- Imóvel de R\$ 50.000 → R\$ 100.000
- Imóvel de R\$ 2.000.000 → R\$ 900.000
- Imóvel de R\$ 500.000 → mantém R\$ 500.000


**Vantagens:**
- ✅ Mantém o tamanho do dataset (não perde observações)
- ✅ Reduz o impacto de valores extremos
- ✅ Preserva a ordenação dos dados
- ✅ Simples de implementar e reverter

**Desvantagens:**
- ❌ Modifica valores reais dos dados
- ❌ Pode mascarar informações importantes
- ❌ A escolha dos percentis é arbitrária

**Quando usar:**
- Quando não podemos perder observações
- Quando outliers são provavelmente erros de medição
- Em modelos sensíveis a valores extremos (regressão linear)

In [ ]:
# @title
# Implementando Winsorização
from scipy.stats.mstats import winsorize

# Criando DataFrame winsorizado
df_winsorized = df_imoveis.copy()

# Aplicando winsorização (limitando a 5% em cada extremo)
for col in numerical_cols:
    df_winsorized[col] = winsorize(df_winsorized[col], limits=(0.05, 0.05))

print("Winsorização aplicada: valores extremos (5% superiores e inferiores) foram limitados")

In [ ]:
# @title Visualizando o impacto das transformações nos outliers
# Vamos comparar as distribuições antes e depois das transformações

# Escolhendo variáveis com distribuições assimétricas para melhor demonstração
vars_to_transform = ['preco', 'idade_anos', 'dist_centro_km']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for i, col in enumerate(vars_to_transform):
    # Original
    axes[i, 0].hist(df_imoveis[col], bins=30, edgecolor='black', alpha=0.7, color='blue')
    axes[i, 0].axvline(df_imoveis[col].mean(), color='red', linestyle='--', label='Média')
    axes[i, 0].axvline(df_imoveis[col].median(), color='green', linestyle='--', label='Mediana')
    axes[i, 0].set_title(f'{col}\n(Original)')
    axes[i, 0].set_ylabel('Frequência')
    if i == 0:
        axes[i, 0].legend()

    # Transformação Log
    axes[i, 1].hist(df_transformed[f'{col}_log'], bins=30, edgecolor='black', alpha=0.7, color='green')
    axes[i, 1].axvline(df_transformed[f'{col}_log'].mean(), color='red', linestyle='--')
    axes[i, 1].axvline(df_transformed[f'{col}_log'].median(), color='green', linestyle='--')
    axes[i, 1].set_title(f'{col}\n(Log)')

    # Transformação Raiz Quadrada
    axes[i, 2].hist(df_transformed[f'{col}_sqrt'], bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[i, 2].axvline(df_transformed[f'{col}_sqrt'].mean(), color='red', linestyle='--')
    axes[i, 2].axvline(df_transformed[f'{col}_sqrt'].median(), color='green', linestyle='--')
    axes[i, 2].set_title(f'{col}\n(Raiz Quadrada)')

plt.suptitle('Impacto das Transformações na Distribuição dos Dados', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Análise do impacto nas métricas
print("=" * 70)
print("ANÁLISE DO IMPACTO DAS TRANSFORMAÇÕES")
print("=" * 70)

for col in vars_to_transform:
    print(f"\n📊 Variável: {col}")
    print("-" * 50)

    # Calculando assimetria (skewness)
    from scipy.stats import skew

    skew_orig = skew(df_imoveis[col])
    skew_log = skew(df_transformed[f'{col}_log'])
    skew_sqrt = skew(df_transformed[f'{col}_sqrt'])

    print(f"Assimetria (Skewness):")
    print(f"  Original: {skew_orig:.3f}")
    print(f"  Log:      {skew_log:.3f} ({'✅ Melhorou' if abs(skew_log) < abs(skew_orig) else '❌ Piorou'})")
    print(f"  Sqrt:     {skew_sqrt:.3f} ({'✅ Melhorou' if abs(skew_sqrt) < abs(skew_orig) else '❌ Piorou'})")

    # Contando outliers (usando IQR)
    def count_outliers_iqr(data):
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)).sum()
        return outliers

    out_orig = count_outliers_iqr(df_imoveis[col])
    out_log = count_outliers_iqr(df_transformed[f'{col}_log'])
    out_sqrt = count_outliers_iqr(df_transformed[f'{col}_sqrt'])

    print(f"\nOutliers detectados (IQR):")
    print(f"  Original: {out_orig}")
    print(f"  Log:      {out_log} ({'✅ Reduziu' if out_log < out_orig else '❌ Aumentou' if out_log > out_orig else '= Manteve'})")
    print(f"  Sqrt:     {out_sqrt} ({'✅ Reduziu' if out_sqrt < out_orig else '❌ Aumentou' if out_sqrt > out_orig else '= Manteve'})")

print("\n💡 INSIGHTS:")
print("• Transformação LOG é mais eficaz para distribuições muito assimétricas (ex: preço)")
print("• Transformação SQRT é mais suave, útil para assimetrias moderadas")
print("• Nem sempre reduz outliers, mas normaliza a distribuição")
print("• A escolha depende do objetivo: normalizar distribuição vs reduzir outliers")

In [ ]:
# @title Comparando histogramas antes e depois da winsorização
plt.figure(figsize=(15, 10))

for i, col in enumerate(numerical_cols[:4], 1):
    plt.subplot(2, 2, i)

    sns.histplot(df_imoveis[col], kde=True, alpha=0.5, color='blue', label='Original')
    sns.histplot(df_winsorized[col], kde=True, alpha=0.5, color='red', label='Winsorizado')

    plt.title(f'Distribuição de {col}')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# @title Verificando a redução no impacto dos outliers com winsorização
import warnings
warnings.filterwarnings("ignore", message=".*partition.*MaskedArray.*")

# Comparando estatísticas descritivas

# Selecionando uma variável para análise detalhada
col_example = 'area_m2'

# Estatísticas originais
stats_orig = df_imoveis[col_example].describe()

# Estatísticas após winsorização
stats_wins = df_winsorized[col_example].describe()

# Comparando
comparison = pd.DataFrame({
    'Original': stats_orig,
    'Winsorizado': stats_wins,
    'Diferença %': ((stats_wins - stats_orig) / stats_orig * 100).round(2)
})

print(f"Comparação de estatísticas para {col_example}:")
print(comparison)

# ==================================================
# MÉTODOS AVANÇADOS (PRÉVIA INFORMATIVA)
# ==================================================

**📚 Nota Importante**: Esta seção é apenas informativa. Os métodos de ML serão
estudados em detalhes a partir da Aula 11 do curso.

## Por que existem métodos de ML para detecção de outliers?

Os métodos estatísticos que vimos (Z-Score, IQR, MAD) funcionam bem para:
- Dados com distribuições conhecidas
- Análise univariada ou bivariada
- Datasets menores

Mas em situações reais, frequentemente encontramos:
- **Dados em alta dimensão** (dezenas ou centenas de variáveis)
- **Padrões complexos** que não seguem distribuições estatísticas simples
- **Outliers contextuais** (normal globalmente, anômalo localmente)

## Exemplos de Métodos Avançados

### 🌲 Isolation Forest
Imagine tentar encontrar uma pessoa específica em uma multidão. Se ela está isolada,
é fácil encontrá-la. Se está no meio da multidão, é difícil. O Isolation Forest
usa essa ideia: outliers são mais fáceis de "isolar" com menos divisões.

### 🎯 Local Outlier Factor (LOF)
Pense em uma festa onde há grupos conversando. Uma pessoa sozinha no canto pode
ser um outlier. Mas também pode ser outlier alguém gritando no meio de um grupo
que conversa baixo. LOF detecta ambos os casos.

### 🔍 DBSCAN
Como um algoritmo de clustering que também identifica pontos que não pertencem
a nenhum grupo - esses são os outliers.

## Quando você usará esses métodos?

- **Aula 11-13**: Introdução ao Machine Learning
- **Aula 15**: Dados desbalanceados e detecção de anomalias
- **Aula 20-21**: Clustering e métodos não-supervisionados
- **Projeto Final**: Aplicação prática em problemas reais

Por agora, os métodos estatísticos que aprendemos são suficientes e fundamentais!

# ==================================================
# 12. IMPACTO DOS TRATAMENTOS
# ==================================================

In [ ]:
# @title
# Exemplo prático: Demonstrando o impacto real dos outliers

# Vamos criar um cenário mais realista adicionando outliers artificiais problemáticos
np.random.seed(42)

# Usando o dataset original
features = ['area_m2', 'quartos', 'banheiros', 'idade_anos', 'garagem_vagas']
X = df_imoveis[features].copy()
y = df_imoveis['preco'].copy()

# ADICIONANDO OUTLIERS ARTIFICIAIS PROBLEMÁTICOS
# Simulando erros de digitação ou problemas de dados
n_outliers = 20
outlier_indices = np.random.choice(len(X), n_outliers, replace=False)

# Criando outliers que quebram a relação linear
for idx in outlier_indices[:10]:
    # Casas pequenas com preços absurdamente altos (erro de digitação)
    X.loc[idx, 'area_m2'] = np.random.uniform(30, 50)  # Área muito pequena
    y.loc[idx] = np.random.uniform(2000000, 5000000)   # Preço muito alto

for idx in outlier_indices[10:]:
    # Casas grandes com preços muito baixos (possível erro)
    X.loc[idx, 'area_m2'] = np.random.uniform(300, 500)  # Área muito grande
    y.loc[idx] = np.random.uniform(50000, 100000)       # Preço muito baixo

print("=" * 60)
print("DEMONSTRAÇÃO DO IMPACTO REAL DE OUTLIERS PROBLEMÁTICOS")
print("=" * 60)
print(f"\n🔍 Adicionados {n_outliers} outliers artificiais que quebram a relação linear")

# MODELO 1: COM OUTLIERS PROBLEMÁTICOS
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_with_outliers = LinearRegression()
model_with_outliers.fit(X_train_scaled, y_train)
y_pred_with = model_with_outliers.predict(X_test_scaled)

r2_with_outliers = r2_score(y_test, y_pred_with)
rmse_with_outliers = np.sqrt(mean_squared_error(y_test, y_pred_with))

# MODELO 2: REMOVENDO OUTLIERS POR IQR
# Identificando e removendo outliers usando IQR
mask_clean = np.ones(len(X), dtype=bool)
for col in features:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask_clean &= (X[col] >= lower) & (X[col] <= upper)

# Também removendo outliers no preço
Q1_price = y.quantile(0.25)
Q3_price = y.quantile(0.75)
IQR_price = Q3_price - Q1_price
mask_clean &= (y >= Q1_price - 1.5 * IQR_price) & (y <= Q3_price + 1.5 * IQR_price)

X_clean = X[mask_clean]
y_clean = y[mask_clean]

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.3, random_state=42
)

X_train_clean_scaled = scaler.fit_transform(X_train_clean)
X_test_clean_scaled = scaler.transform(X_test_clean)

model_without_outliers = LinearRegression()
model_without_outliers.fit(X_train_clean_scaled, y_train_clean)
y_pred_without = model_without_outliers.predict(X_test_clean_scaled)

r2_without_outliers = r2_score(y_test_clean, y_pred_without)
rmse_without_outliers = np.sqrt(mean_squared_error(y_test_clean, y_pred_without))

# MODELO 3: USANDO WINSORIZAÇÃO
X_wins = X.copy()
y_wins = y.copy()

# Winsorização das features
for col in features:
    from scipy.stats.mstats import winsorize
    X_wins[col] = winsorize(X_wins[col], limits=(0.05, 0.05))

# Winsorização do preço
y_wins = pd.Series(winsorize(y_wins, limits=(0.05, 0.05)))

X_train_wins, X_test_wins, y_train_wins, y_test_wins = train_test_split(
    X_wins, y_wins, test_size=0.3, random_state=42
)

X_train_wins_scaled = scaler.fit_transform(X_train_wins)
X_test_wins_scaled = scaler.transform(X_test_wins)

model_winsorized = LinearRegression()
model_winsorized.fit(X_train_wins_scaled, y_train_wins)
y_pred_wins = model_winsorized.predict(X_test_wins_scaled)

r2_winsorized = r2_score(y_test_wins, y_pred_wins)
rmse_winsorized = np.sqrt(mean_squared_error(y_test_wins, y_pred_wins))

# COMPARAÇÃO DOS RESULTADOS
print("\n📊 RESULTADOS:")
print("-" * 50)
print(f"COM OUTLIERS PROBLEMÁTICOS:")
print(f"  R² Score: {r2_with_outliers:.4f}")
print(f"  RMSE: R$ {rmse_with_outliers:,.2f}")

print(f"\nAPÓS REMOÇÃO DE OUTLIERS (IQR):")
print(f"  R² Score: {r2_without_outliers:.4f}")
print(f"  RMSE: R$ {rmse_without_outliers:,.2f}")
print(f"  Dados removidos: {len(X) - len(X_clean)} ({(1 - len(X_clean)/len(X))*100:.1f}%)")

print(f"\nCOM WINSORIZAÇÃO:")
print(f"  R² Score: {r2_winsorized:.4f}")
print(f"  RMSE: R$ {rmse_winsorized:,.2f}")
print(f"  Dados mantidos: 100%")

print(f"\n📈 MELHORIAS:")
print(f"  Remoção vs Original: R² melhorou {((r2_without_outliers - r2_with_outliers)/abs(r2_with_outliers)*100):.1f}%")
print(f"  Winsorização vs Original: R² melhorou {((r2_winsorized - r2_with_outliers)/abs(r2_with_outliers)*100):.1f}%")

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Com outliers
axes[0].scatter(y_test, y_pred_with, alpha=0.5, color='red')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_xlabel('Preço Real')
axes[0].set_ylabel('Preço Previsto')
axes[0].set_title(f'Com Outliers\n(R²={r2_with_outliers:.3f})')
axes[0].grid(True, alpha=0.3)

# Plot 2: Sem outliers
axes[1].scatter(y_test_clean, y_pred_without, alpha=0.5, color='green')
axes[1].plot([y_test_clean.min(), y_test_clean.max()],
             [y_test_clean.min(), y_test_clean.max()], 'k--', lw=2)
axes[1].set_xlabel('Preço Real')
axes[1].set_ylabel('Preço Previsto')
axes[1].set_title(f'Após Remoção IQR\n(R²={r2_without_outliers:.3f})')
axes[1].grid(True, alpha=0.3)

# Plot 3: Com winsorização
axes[2].scatter(y_test_wins, y_pred_wins, alpha=0.5, color='blue')
axes[2].plot([y_test_wins.min(), y_test_wins.max()],
             [y_test_wins.min(), y_test_wins.max()], 'k--', lw=2)
axes[2].set_xlabel('Preço Real')
axes[2].set_ylabel('Preço Previsto')
axes[2].set_title(f'Com Winsorização\n(R²={r2_winsorized:.3f})')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Impacto do Tratamento de Outliers na Regressão Linear', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 CONCLUSÃO:")
print("Os outliers artificiais (erros de dados) prejudicaram significativamente o modelo.")
print("Tanto a remoção quanto a winsorização melhoraram o desempenho, com trade-offs:")
print("• Remoção: Melhor R², mas perde dados")
print("• Winsorização: Mantém todos os dados, melhoria moderada")

## 🎯 Micro-atividade: Outlier Detective - Atividade Guiada (10 min)

**Objetivo**: Comparar métodos de detecção e escolher tratamento apropriado

**Cenário**: Você é um cientista de dados analisando preços de imóveis. Precisa identificar e tratar outliers!

### Parte 1: Execute o código abaixo para detectar outliers

In [ ]:
# @title 🕵️ Detecção de Outliers - Código Guiado

# Gerando dados de exemplo (preços de imóveis)
np.random.seed(42)
n_samples = 200

# Preços normais
precos_normais = np.random.normal(500000, 100000, n_samples-10)

# Adicionando outliers intencionais
outliers_baixo = np.array([50000, 75000])  # Muito baratos
outliers_alto = np.array([2000000, 2500000, 3000000, 5000000])  # Muito caros
outliers_extremo = np.array([10000000, 15000000, 20000000, 25000000])  # Extremamente caros

# Combinando todos os preços
precos = np.concatenate([precos_normais, outliers_baixo, outliers_alto, outliers_extremo])
df_exemplo = pd.DataFrame({'preco': precos})

print('📊 ANÁLISE DOS DADOS DE IMÓVEIS')
print('=' * 50)
print(f'Total de imóveis: {len(df_exemplo)}')
print(f'Preço médio: R$ {df_exemplo["preco"].mean():,.2f}')
print(f'Preço mediano: R$ {df_exemplo["preco"].median():,.2f}')
print(f'Desvio padrão: R$ {df_exemplo["preco"].std():,.2f}')

# Método 1: Z-Score
z_scores_ex = np.abs(stats.zscore(df_exemplo['preco']))
outliers_zscore = df_exemplo[z_scores_ex > 3]

# Método 2: IQR
Q1 = df_exemplo['preco'].quantile(0.25)
Q3 = df_exemplo['preco'].quantile(0.75)
IQR = Q3 - Q1
outliers_iqr = df_exemplo[(df_exemplo['preco'] < Q1 - 1.5*IQR) | (df_exemplo['preco'] > Q3 + 1.5*IQR)]

# Método 3: Percentis
outliers_percentil = df_exemplo[(df_exemplo['preco'] < df_exemplo['preco'].quantile(0.01)) |
                                (df_exemplo['preco'] > df_exemplo['preco'].quantile(0.99))]

print('\n🔍 OUTLIERS DETECTADOS:')
print('-' * 50)
print(f'Z-Score (|z| > 3): {len(outliers_zscore)} outliers')
print(f'IQR (1.5 * IQR): {len(outliers_iqr)} outliers')
print(f'Percentis (1% e 99%): {len(outliers_percentil)} outliers')

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Boxplot
axes[0].boxplot(df_exemplo['preco'])
axes[0].set_title('Boxplot dos Preços')
axes[0].set_ylabel('Preço (R$)')

# Histograma
axes[1].hist(df_exemplo['preco'], bins=30, edgecolor='black')
axes[1].axvline(df_exemplo['preco'].mean(), color='red', linestyle='--', label='Média')
axes[1].axvline(df_exemplo['preco'].median(), color='green', linestyle='--', label='Mediana')
axes[1].set_title('Distribuição dos Preços')
axes[1].set_xlabel('Preço (R$)')
axes[1].legend()

# Scatter com outliers marcados
axes[2].scatter(range(len(df_exemplo)), df_exemplo['preco'], alpha=0.5)
axes[2].scatter(outliers_iqr.index, outliers_iqr['preco'], color='red', s=50, label='Outliers (IQR)')
axes[2].set_title('Identificação de Outliers')
axes[2].set_xlabel('Índice')
axes[2].set_ylabel('Preço (R$)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# @title 💡 Tratamento de Outliers - Comparação de Métodos

print('💡 COMPARAÇÃO DE MÉTODOS DE TRATAMENTO')
print('=' * 50)

# Método 1: Remoção
df_sem_outliers = df_exemplo[z_scores_ex <= 3].copy()

# Método 2: Capping (limitação)
df_capped = df_exemplo.copy()
lower_cap = df_exemplo['preco'].quantile(0.05)
upper_cap = df_exemplo['preco'].quantile(0.95)
df_capped['preco'] = df_capped['preco'].clip(lower=lower_cap, upper=upper_cap)

# Método 3: Transformação Log
df_log = df_exemplo.copy()
df_log['preco_log'] = np.log1p(df_log['preco'])  # log1p para evitar log(0)

# Comparando estatísticas
comparacao = pd.DataFrame({
    'Original': [df_exemplo['preco'].mean(), df_exemplo['preco'].std(), len(df_exemplo)],
    'Sem Outliers': [df_sem_outliers['preco'].mean(), df_sem_outliers['preco'].std(), len(df_sem_outliers)],
    'Com Capping': [df_capped['preco'].mean(), df_capped['preco'].std(), len(df_capped)],
}, index=['Média', 'Desvio Padrão', 'N amostras'])

print('\n📊 Comparação dos Métodos:')
print(comparacao.round(2))

# Visualização dos resultados
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original
axes[0, 0].hist(df_exemplo['preco'], bins=30, edgecolor='black', color='blue', alpha=0.7)
axes[0, 0].set_title('Dados Originais')
axes[0, 0].set_xlabel('Preço (R$)')

# Sem outliers
axes[0, 1].hist(df_sem_outliers['preco'], bins=30, edgecolor='black', color='green', alpha=0.7)
axes[0, 1].set_title(f'Após Remoção (n={len(df_sem_outliers)})')
axes[0, 1].set_xlabel('Preço (R$)')

# Com capping
axes[1, 0].hist(df_capped['preco'], bins=30, edgecolor='black', color='orange', alpha=0.7)
axes[1, 0].set_title('Após Capping (5%-95%)')
axes[1, 0].set_xlabel('Preço (R$)')

# Transformação Log
axes[1, 1].hist(df_log['preco_log'], bins=30, edgecolor='black', color='purple', alpha=0.7)
axes[1, 1].set_title('Após Transformação Log')
axes[1, 1].set_xlabel('Log(Preço)')

plt.tight_layout()
plt.show()

print('\n🎯 RECOMENDAÇÃO:')
print('-' * 50)
print('Para dados de imóveis, o método de CAPPING é geralmente preferível porque:')
print('✅ Mantém todos os dados (não perde informação)')
print('✅ Preserva o tamanho da amostra')
print('✅ Limita valores extremos sem distorcer completamente')
print('✅ Mantém interpretabilidade (ainda em R$)')
print('\n⚠️ A remoção pode ser apropriada se os outliers forem erros de digitação')
print('⚠️ A transformação log é útil quando a relação é multiplicativa')

### Parte 2: Análise e Decisão 🎯

**Perguntas para reflexão:**

1. **Detecção**: Qual método detectou mais outliers? Por quê?

2. **Impacto**: Como a média e mediana mudaram após cada tratamento?

3. **Contexto de negócio**: Para imóveis de luxo, remover outliers altos faz sentido?

4. **Trade-offs**:
   - Remoção: Perde dados mas limpa distribuição
   - Capping: Mantém dados mas altera valores
   - Log: Normaliza mas perde interpretabilidade

**💡 Insight importante**:
Nem todo outlier é um erro! Em imóveis:
- Outliers baixos podem ser imóveis com problemas
- Outliers altos podem ser mansões ou coberturas de luxo
- A decisão depende do objetivo da análise!

### ARMADILHAS COMUNS A EVITAR

1. **Viés de exclusão seletiva**
   - Remover apenas outliers que "atrapalham" suas hipóteses
   - Tratamento inconsistente entre grupos de comparação

2. **Aplicar regras rígidas sem contexto**
   - Remover cegamente baseado em critérios estatísticos (ex: todos Z > 3)
   - Ignorar o conhecimento do domínio e significado dos dados

3. **Ignorar outliers multivariados**
   - Focar apenas em distribuições univariadas
   - Perder anomalias que só aparecem em combinações de variáveis

4. **Aplicar transformações sem considerar interpretabilidade**
   - Dificultar a interpretação dos resultados
   - Esquecer de reverter transformações ao reportar resultados

5. **Não avaliar o impacto das decisões**
   - Deixar de comparar resultados antes e depois do tratamento
   - Não testar a sensibilidade dos resultados a diferentes métodos